# Biohub - Cell Tracking: DoG Detection + Trackastra Graph Transformer

**Competition:** [Biohub - Cell Tracking During Development](https://www.kaggle.com/competitions/biohub-cell-tracking-during-development)

## Problem

Track individual cells through 3D fluorescence microscopy of developing zebrafish embryos.
Each dataset is a 4D volume (time × Z × Y × X). The goal is to output:
- **Nodes**: one detected centroid per cell per timepoint (integer voxel coords)
- **Edges**: temporal links connecting a cell at time *t* to the same cell at time *t+1*
- **Divisions**: a node with two outgoing edges = one parent cell splitting into two daughters

**Metric:** Edge Jaccard (predicted links vs ground-truth links) + Division Jaccard,
weight-averaged across datasets. Cells are sparsely labelled — only a subset of real
cells appear in ground truth, so over-detecting is penalised via a node-count adjustment.

## Strategy: DoG + Trackastra

**Detection** — Difference of Gaussians (DoG) band-pass filter on an isotropic-resampled
volume, with intensity-weighted centre-of-mass (COM) refinement and physical-space NMS.
Classic, fast, no GPU required.

**Tracking** — [Trackastra](https://github.com/weigertlab/trackastra) (ECCV 2024),
winner of the ISBI 2024 Cell Tracking Challenge. A graph transformer that predicts
association weights between all candidate cell pairs in a sliding temporal window and
solves the assignment globally — no hand-tuned distance gates, divisions handled natively.

**Why not Hungarian/ILP baseline?** Frame-pair Hungarian matching requires a hand-tuned
max-distance gate and cannot handle divisions. Trackastra learns both from CTC data.

**Pipeline:**
1. Correct anisotropy (XY block-mean ÷4 → ~isotropic 1.625 µm voxels)
2. DoG multi-scale detection on isotropic volume → centroid list per timepoint
3. Paint each centroid as a labelled ball mask (Trackastra input format)
4. `Trackastra.from_pretrained("ctc").track(imgs, masks, mode="greedy")`
5. Parse `track_graph` nodes + edges → submission CSV

## Setup

### Package strategy

Kaggle notebooks run **internet-disabled** at submission time. Packages must be
pre-downloaded and bundled as a Kaggle dataset input.

**Step 1** (internet enabled, run once): download all wheels to `packages/`.
numpy and scipy are **pre-installed** on Kaggle — we capture their exact versions
and pin them in the download so pip does not pull a different (incompatible) version
of either when resolving trackastra's dependency tree.
Without pinning, scipy ≥1.14 can trigger an `AttributeError: _blas_supports_fpe`
crash because it expects a numpy ≥2.1 internal symbol that may not be present.

**Step 2** (internet disabled, runs on submission): install from local `packages/`
with the same pins. Falls back to PyPI if the packages folder is missing (local dev).

In [1]:
# Step 1 — download wheels once (internet enabled). numpy/scipy pre-installed on Kaggle; pin them
# so trackastra's dependency resolver cannot upgrade them and cause _blas_supports_fpe mismatch.
# ! NP=$(python3 -c "import numpy; print(numpy.__version__)") && \
#   SP=$(python3 -c "import scipy; print(scipy.__version__)") && \
#   pip download -q "numpy==$NP" "scipy==$SP" trackastra zarr blosc2 scikit-image pandas tqdm -d packages/

In [2]:
# Step 2 — install (offline-safe). Pinned numpy/scipy prevent silent upgrades.
! NP=$(python3 -c "import numpy; print(numpy.__version__)") && \
  SP=$(python3 -c "import scipy; print(scipy.__version__)") && \
  pip install -q --no-index --find-links "/kaggle/input/models/jirkaborovec/biohubcelltrack-trackastra-artifact-and-packages/pytorch/default/1/packages/packages" "numpy==$NP" "scipy==$SP" trackastra zarr blosc2 scikit-image pandas tqdm \
  || pip install -q "numpy==$NP" "scipy==$SP" trackastra zarr blosc2 scikit-image pandas tqdm
! pip list | grep -E 'trackastra|numpy|scipy|zarr'

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
libcuml-cu12 26.2.0 requires cuda-toolkit[cublas,cufft,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.
cuml-cu12 26.2.0 requires cuda-toolkit[cublas,cufft,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
torchaudio 2.10.0+cu128 requires torch==2.10.0, but you have torch 2.12.1 which is incompatible.
cudf-cu12 26.2.1 requires cuda-toolkit[nvcc,nvrtc]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 re

In [3]:
# Step 3 — download Trackastra model weights once (internet enabled).
# Saves "ctc" checkpoint to models/ctc/. Attach that folder as a Kaggle dataset input.
# import shutil
# from pathlib import Path
# import platformdirs
# from trackastra.model import Trackastra

# dst = Path("models/ctc")
# if not dst.exists():
#     Trackastra.from_pretrained("ctc")  # downloads to user_data_dir/models/ctc/
#     cache = Path(platformdirs.user_data_dir("trackastra")) / "models" / "ctc"
#     shutil.copytree(cache, dst)
#     size_mb = sum(f.stat().st_size for f in dst.rglob("*") if f.is_file()) / 1e6
#     print(f"Saved to {dst}  ({size_mb:.1f} MB)")
# else:
#     print(f"Already exists: {dst}")

In [4]:
# Step 4 — zip pip wheels for Kaggle dataset upload, then remove the source folder.
# Upload packages.zip as a Kaggle dataset; attach as input on submission.
# ! zip -qr packages.zip packages/ && rm -rf packages/
# ! ls -lh packages.zip

## Imports + Constants

Key library roles:
- `zarr` / `blosc2` — read the competition's zarr v3 compressed volumes
- `scipy.ndimage.gaussian_filter` — DoG band-pass filtering
- `scipy.spatial.cKDTree` — fast radius-search for physical-space NMS
- `skimage.feature.peak_local_max` — local maxima extraction after DoG
- `trackastra` — graph-transformer tracker (loaded after install above)
- `contextlib.redirect_stderr` — silences Trackastra's tqdm progress bars
  (they write to stderr; redirect captures all output cleanly without patching internals)

In [5]:
from __future__ import annotations

import contextlib
import io
import json
import logging
import os
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy.ndimage import gaussian_filter
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree
from skimage.feature import peak_local_max
from skimage.morphology import ball, binary_dilation
from skimage.segmentation import watershed
from tqdm.auto import tqdm

logging.getLogger("trackastra").setLevel(logging.WARNING)

# Paths — override via env vars for local development
DATA_DIR = Path(os.environ.get("KAGGLE_DATA_DIR", "/kaggle/input/competitions/biohub-cell-tracking-during-development"))
OUTPUT_DIR = Path(os.environ.get("KAGGLE_OUTPUT_DIR", "."))  # "." = /kaggle/working/ on Kaggle
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"Train exists: {TRAIN_DIR.exists()}, Test exists: {TEST_DIR.exists()}")

Device: cuda
Train exists: True, Test exists: True


## Data Loading

The competition uses **zarr v3** format with **blosc2** compression, one chunk per
timepoint. Layout on disk:
```
<dataset>.zarr/
  0/                  ← array "0" (the image)
    zarr.json         ← metadata: shape, dtype, chunk shape
    c/
      <t>/0/0/0       ← one blosc2-compressed chunk per timepoint t
```

`load_timepoint_blosc2` reads the raw bytes and decompresses directly — faster than
going through the zarr library's array interface which adds Python overhead per chunk.
A zarr fallback handles any edge cases (non-standard chunking, future format changes).

In [6]:
def get_volume_shape(zarr_path: Path) -> tuple[int, int, int, int]:
    """Return (T, Z, Y, X) from zarr.json metadata — no zarr import needed."""
    with (zarr_path / "0" / "zarr.json").open() as f:
        return tuple(json.load(f)["shape"])


def load_timepoint_blosc2(zarr_path: Path, t: int) -> np.ndarray:
    """Load one timepoint (Z, Y, X) directly from blosc2 chunk.

    Chunk path: <zarr>/0/c/<t>/0/0/0. Faster than zarr array slicing.
    """
    import blosc2

    chunk_path = zarr_path / "0" / "c" / str(t) / "0" / "0" / "0"
    raw = chunk_path.read_bytes()
    return np.frombuffer(blosc2.decompress(raw), dtype=np.uint16).reshape(-1)


def load_timepoint(zarr_path: Path, t: int, shape_zyx: tuple[int, int, int]) -> np.ndarray:
    """Load one timepoint with blosc2 fast path and zarr fallback."""
    try:
        flat = load_timepoint_blosc2(zarr_path, t)
        n = shape_zyx[0] * shape_zyx[1] * shape_zyx[2]
        return flat[:n].reshape(shape_zyx).astype(np.float32)
    except Exception:
        import zarr as _zarr
        return _zarr.open(str(zarr_path), mode="r")["0"][t].astype(np.float32)

## Anisotropy Correction

Confocal microscopes have much lower axial (Z) resolution than lateral (XY) resolution.
In this dataset:

| Axis | µm/voxel |
|------|----------|
| Z    | 1.625    |
| Y    | 0.40625  |
| X    | 0.40625  |

Z spacing is **4× coarser** than XY. This matters because:
- DoG `gaussian_filter` applies the same σ in every axis — distances must be isotropic
- Trackastra was trained on isotropic volumes (CTC benchmark data)
- Physical NMS radius comparisons require isotropic coordinates

**Fix:** block-mean average XY by 4 → new voxel size z=y=x≈1.625 µm.
This is lossless for blob detection at cell scale (~8 µm diameter) and 4× cheaper
than upsampling Z.

In [7]:
# Physical voxel scale (µm/voxel)
VOXEL_Z = 1.625
VOXEL_Y = 0.40625
VOXEL_X = 0.40625
XY_DOWNSAMPLE = 4          # block-mean XY to get ~isotropic 1.625 µm grid
SCALE_ZYX = np.array([VOXEL_Z, VOXEL_Y, VOXEL_X])


def make_isotropic(vol: np.ndarray) -> np.ndarray:
    """Block-average XY by XY_DOWNSAMPLE to get isotropic voxels.

    Raw: z=1.625, y=x=0.406 µm. After ÷4: z=y=x=1.625 µm.
    """
    nz, ny, nx = vol.shape
    ny2 = ny // XY_DOWNSAMPLE
    nx2 = nx // XY_DOWNSAMPLE
    cropped = vol[:, : ny2 * XY_DOWNSAMPLE, : nx2 * XY_DOWNSAMPLE]
    return cropped.reshape(nz, ny2, XY_DOWNSAMPLE, nx2, XY_DOWNSAMPLE).mean(axis=(2, 4))

## DoG Cell Detection

### What is DoG?

Difference of Gaussians (DoG) is a **band-pass filter** that highlights structures
at a specific spatial scale while suppressing background and noise:

```
DoG(σ) = Gaussian(σ) − Gaussian(σ × k)
```

Positive DoG response = blob-like bright structure with radius ≈ σ√2.
It approximates the Laplacian of Gaussian (LoG) at much lower cost.

### Multi-scale detection

Cells vary in apparent size across the volume (depth-dependent PSF, varying cell sizes).
We run DoG at three scales and **union the peak lists** before NMS:

| σ (iso voxels) | ≈ cell diameter (µm) |
|---|---|
| 1.0 | ~2.8 (small/young cells) |
| 1.8 | ~5.1 (typical) |
| 3.0 | ~8.5 (large/dividing cells) |

### Centre-of-mass (COM) refinement

`peak_local_max` returns integer voxel coords. COM refinement shifts each peak to the
intensity-weighted centroid in a local window — sub-voxel accuracy, important for the
metric's ≤7 µm matching threshold.
The window is asymmetric (tighter in Z) to match the PSF shape in original space.

### Physical-space NMS

Peaks from different scales often detect the same cell. We deduplicate by converting
coords to µm, building a kd-tree, and suppressing lower-score peaks within 4 µm of
a stronger peak.

In [8]:
DOG_SIGMAS = (1.0, 1.8, 3.0)
DOG_K = 1.6          # sigma_high = sigma_low × DOG_K
DOG_THR_PCT = 80.0   # keep peaks above this percentile of positive DoG response
REFINE_RZ = 2        # COM half-window in Z (original voxels)
REFINE_RYX = 5       # COM half-window in Y/X (original voxels)
MIN_PEAK_DIST = 2    # min peak separation in isotropic voxels
NMS_RADIUS_UM = 4.0  # physical dedup radius (µm) — ~half typical cell diameter


def _dog_scale_back(pk_iso: np.ndarray) -> np.ndarray:
    """Convert isotropic peak coords to original-voxel float coords.

    Places y, x at the block centre (not edge).
    """
    out = pk_iso.astype(float)
    out[:, 1] = out[:, 1] * XY_DOWNSAMPLE + (XY_DOWNSAMPLE - 1) / 2.0
    out[:, 2] = out[:, 2] * XY_DOWNSAMPLE + (XY_DOWNSAMPLE - 1) / 2.0
    return out


def _com_refine_orig(vol: np.ndarray, zyx: np.ndarray) -> np.ndarray:
    """Intensity-weighted COM refinement in original anisotropic volume.

    Asymmetric window: REFINE_RZ in Z, REFINE_RYX in Y/X (~spherical in µm).
    """
    nz, ny, nx = vol.shape
    z, y, x = int(round(zyx[0])), int(round(zyx[1])), int(round(zyx[2]))
    z0, z1 = max(0, z - REFINE_RZ), min(nz, z + REFINE_RZ + 1)
    y0, y1 = max(0, y - REFINE_RYX), min(ny, y + REFINE_RYX + 1)
    x0, x1 = max(0, x - REFINE_RYX), min(nx, x + REFINE_RYX + 1)
    patch = vol[z0:z1, y0:y1, x0:x1].astype(np.float64)
    w = np.maximum(patch - float(patch.min()), 0.0)
    total = float(w.sum())
    if total < 1e-12:
        return zyx.copy()
    gz, gy, gx = np.mgrid[z0:z1, y0:y1, x0:x1]
    return np.array([(gz * w).sum(), (gy * w).sum(), (gx * w).sum()]) / total


def _nms_physical(
    coords: np.ndarray, scores: np.ndarray, radius_um: float
) -> tuple[np.ndarray, np.ndarray]:
    """Non-maximum suppression in physical µm space.

    Suppresses all lower-score peaks within radius_um of a stronger peak.
    """
    if len(coords) <= 1:
        return coords, scores
    pts = coords * SCALE_ZYX[None, :]
    order = np.argsort(-scores)
    tree = cKDTree(pts)
    killed = np.zeros(len(coords), dtype=bool)
    keep: list[int] = []
    for i in order:
        if killed[i]:
            continue
        keep.append(int(i))
        killed[tree.query_ball_point(pts[i], r=radius_um)] = True
    k = np.array(keep)
    return coords[k], scores[k]


def detect_peaks_dog(
    vol: np.ndarray,
    dog_thr_pct: float = DOG_THR_PCT,
    topk: int | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    """Multi-scale DoG detection on raw anisotropic volume.

    Returns:
        coords: (N, 3) float array in original voxel space (z, y, x).
        scores: (N,) normalised DoG response scores.
    """
    pooled = make_isotropic(vol)
    all_coords: list[np.ndarray] = []
    all_scores: list[float] = []

    for sigma in DOG_SIGMAS:
        dog = gaussian_filter(pooled, sigma) - gaussian_filter(pooled, sigma * DOG_K)
        pos_vals = dog[dog > 0]
        if pos_vals.size == 0:
            continue
        thr = float(np.percentile(pos_vals, dog_thr_pct))
        iso_peaks = peak_local_max(
            dog, min_distance=MIN_PEAK_DIST, threshold_abs=thr, exclude_border=False
        )
        if len(iso_peaks) == 0:
            continue
        resp = dog[iso_peaks[:, 0], iso_peaks[:, 1], iso_peaks[:, 2]].astype(float)
        resp /= max(float(resp.max()), 1e-6)
        orig_init = _dog_scale_back(iso_peaks)
        for p, r in zip(orig_init, resp, strict=False):
            all_coords.append(_com_refine_orig(vol, p))
            all_scores.append(float(r))

    if not all_coords:
        return np.zeros((0, 3), dtype=float), np.zeros(0, dtype=float)

    coords = np.array(all_coords)
    scores = np.array(all_scores)
    coords, scores = _nms_physical(coords, scores, NMS_RADIUS_UM)

    if topk is not None and len(coords) > topk:
        best = np.argsort(-scores)[: int(topk)]
        coords, scores = coords[best], scores[best]

    return coords, scores

## Instance Mask Builder

Trackastra does **not** accept raw point coordinates — it requires **instance
segmentation masks**: integer-labelled 3D arrays where each connected component is
one cell (background = 0, cell 1 = 1, cell 2 = 2, …).

We have two ways to turn DoG centroids into masks, switchable via `MASK_MODE` below:

**`"watershed"` (default) — real segmentation.** Marker-controlled watershed seeded by
the DoG centroids, grown on the actual intensity volume. Each cell gets its *true*
shape and brightness profile. This matters because Trackastra is a graph transformer
that learns to associate cells from their **appearance and morphology** — the whole
reason it beats nearest-neighbour linking. Feeding it featureless spheres throws that
signal away; real masks give it back. Each basin is capped to a sphere of
`MAX_CELL_RADIUS_ISO` around its seed so a bright neighbour cannot flood the frame.

**`"ball"` — fixed spheres.** Paints a small ball around each centroid. Cheap and
deterministic, but morphology-free — kept as a baseline to A/B against watershed via
the proxy scorer.

In [9]:
BALL_RADIUS_ISO = 3       # sphere radius in isotropic voxels (~4.9 µm) — ball mode
MAX_CELL_RADIUS_ISO = 5   # watershed basin cap in isotropic voxels (~8 µm — one cell)
SEG_FG_PCT = 75.0         # intensity percentile: watershed grows only into brighter voxels


def centroids_to_mask(
    shape_zyx_iso: tuple[int, int, int],
    centroids_orig: list[tuple[float, float, float]],
    radius: int = BALL_RADIUS_ISO,
) -> np.ndarray:
    """Create uint16 ball-instance mask from original-voxel centroid list.

    Args:
        shape_zyx_iso: (Z, Y_iso, X_iso) isotropic volume shape.
        centroids_orig: List of (z, y, x) float coords in original voxel space.
        radius: Ball radius in isotropic voxels.

    Returns:
        uint16 mask (Z, Y_iso, X_iso); background=0, cells=1,2,...,N.
    """
    nz, ny, nx = shape_zyx_iso
    mask = np.zeros((nz, ny, nx), dtype=np.uint16)

    for cell_id, (z_o, y_o, x_o) in enumerate(centroids_orig, start=1):
        zi = int(round(z_o))
        yi = int(round(y_o / XY_DOWNSAMPLE))
        xi = int(round(x_o / XY_DOWNSAMPLE))

        z0, z1 = max(0, zi - radius), min(nz, zi + radius + 1)
        y0, y1 = max(0, yi - radius), min(ny, yi + radius + 1)
        x0, x1 = max(0, xi - radius), min(nx, xi + radius + 1)

        gz, gy, gx = np.ogrid[z0:z1, y0:y1, x0:x1]
        sphere = ((gz - zi) ** 2 + (gy - yi) ** 2 + (gx - xi) ** 2) <= radius ** 2
        patch = mask[z0:z1, y0:y1, x0:x1]
        patch[sphere & (patch == 0)] = np.uint16(cell_id)

    return mask


def centroids_to_seg_mask(
    vol_iso: np.ndarray,
    centroids_orig: list[tuple[float, float, float]],
    max_radius: int = MAX_CELL_RADIUS_ISO,
    fg_pct: float = SEG_FG_PCT,
) -> np.ndarray:
    """Marker-controlled watershed instance mask seeded by DoG centroids.

    Grows a real cell-shaped region per centroid on the isotropic intensity volume,
    giving Trackastra genuine appearance/morphology features (unlike fixed balls).
    Each basin is bounded to a sphere of ``max_radius`` around its seed and to voxels
    brighter than the ``fg_pct`` intensity percentile, so a bright neighbour cannot
    swallow the frame.

    Args:
        vol_iso: Isotropic intensity volume (Z, Y_iso, X_iso) — same array fed to the tracker.
        centroids_orig: List of (z, y, x) float coords in original voxel space.
        max_radius: Basin radius cap in isotropic voxels.
        fg_pct: Intensity percentile; watershed grows only into voxels above it.

    Returns:
        uint16 mask (Z, Y_iso, X_iso); background=0, cells=1,2,...,N.
    """
    nz, ny, nx = vol_iso.shape
    markers = np.zeros((nz, ny, nx), dtype=np.int32)
    for cell_id, (z_o, y_o, x_o) in enumerate(centroids_orig, start=1):
        zi = min(max(int(round(z_o)), 0), nz - 1)
        yi = min(max(int(round(y_o / XY_DOWNSAMPLE)), 0), ny - 1)
        xi = min(max(int(round(x_o / XY_DOWNSAMPLE)), 0), nx - 1)
        markers[zi, yi, xi] = cell_id

    if not centroids_orig:
        return markers.astype(np.uint16)

    seed_mask = markers > 0
    bound = binary_dilation(seed_mask, ball(max_radius))         # spatial cap per cell
    fg = ((vol_iso > np.percentile(vol_iso, fg_pct)) & bound) | seed_mask
    labels = watershed(-vol_iso, markers, mask=fg, compactness=0.01)
    return labels.astype(np.uint16)

## Load Trackastra Model

### What is Trackastra?

Trackastra (Löffler et al., ECCV 2024) frames cell tracking as a **graph problem**:
- **Nodes** = detected cells at each timepoint
- **Candidate edges** = all pairs of cells within a temporal window
- A **graph transformer** predicts a link-probability score for every candidate edge
- The final tracking graph is assembled by solving an assignment from these scores

Key advantages over classical Hungarian matching:
1. **Global**: considers all frames in a window jointly, not frame-pairs independently
2. **Divisions**: a node can have two outgoing edges (parent → daughter1, daughter2)
3. **Learned distances**: handles appearance/occlusion, not just Euclidean proximity
4. **No hand-tuning**: max-distance gate, gap-closing, division penalty all learned

### The `ctc` pretrained model

Trackastra ships several pretrained checkpoints. `"ctc"` is the only one trained on
3D data — it was trained on all publicly available Cell Tracking Challenge (CTC)
datasets (2D and 3D) with ground-truth annotations. It won the ISBI 2024 CTC.

### Offline bundling for Kaggle submission

Kaggle disables internet at submission time. **Step 3** in the Setup section above
downloads the weights to `models/ctc/` alongside `packages/`. Attach the `models/`
folder as a Kaggle dataset input; the cell below loads from that path automatically.

Path resolution order:
1. `models/ctc/` — bundled dataset (Kaggle offline submission)
2. `from_pretrained("ctc")` — online download (local dev only; fails on Kaggle submit)

In [10]:
from trackastra.model import Trackastra

MODEL_DIR = Path("/kaggle/input/models/jirkaborovec/biohubcelltrack-trackastra-artifact-and-packages/pytorch/default/1/models/ctc")  # bundled weights from Setup Step 3

if MODEL_DIR.exists():
    model = Trackastra.from_folder(MODEL_DIR, device=DEVICE)
    print(f"Loaded Trackastra from {MODEL_DIR}")
else:
    # Online fallback — only works during dev; internet disabled on Kaggle submission
    model = Trackastra.from_pretrained("ctc", device=DEVICE)
    print("Loaded Trackastra from pretrained hub (online)")

Loaded Trackastra from /kaggle/input/models/jirkaborovec/biohubcelltrack-trackastra-artifact-and-packages/pytorch/default/1/models/ctc


## Submission Format

The competition CSV has **two interleaved row types**:

| `row_type` | Required fields | Placeholder fields |
|---|---|---|
| `node` | `node_id`, `t`, `z`, `y`, `x` | `source_id=-1`, `target_id=-1` |
| `edge` | `source_id`, `target_id` (referencing `node_id`) | `node_id=-1`, `t=-1`, `z=-1`, `y=-1`, `x=-1` |

**Key rules:**
- `node_id` must be unique across the entire submission (not just within one dataset)
- `z`, `y`, `x` are integer voxel coords in the **original** (anisotropic) space
- `dataset` must exactly match the zarr folder stem (e.g. `44b6_0113de3b`, not `44b6`)
- Every test dataset must appear; missing dataset = score 0 for that dataset
- `id` is a throwaway consecutive integer index (no semantic meaning)

We use frozen dataclasses (defined with the tracking function below, so the two always
run together) to catch field mismatches at construction time.

## Inference Loop

### Per-dataset processing

For each test zarr:
1. **Load all T timepoints** into a 4D `float32` array `imgs_4d` (T, Z, Y_iso, X_iso)
   and a 4D `uint16` mask array `masks_4d` of the same shape.
2. **DoG detection** per frame → centroid list in original voxel space.
   Centroids are also stored in `det_lookup[(t, label)]` for post-tracking lookup.
3. **Paint masks** from centroids via `centroids_to_mask`.
4. **Trackastra** receives the full 4D arrays and returns a `track_graph`
   (a `networkx.DiGraph`).

### Reading the track graph

`track_graph` nodes have attributes:
- `"time"` (int): timepoint index
- `"label"` (int): the cell's label in `masks_4d` at that timepoint
  — this is the `cell_id` we used in `centroids_to_mask`, so it indexes `det_lookup`

`track_graph` edges go from earlier to later node. A node with **two outgoing edges**
is a division event (parent → two daughters).

### Global node IDs

`node_id` in the submission must be globally unique across all datasets. We use a
monotonically increasing counter (`global_node_id`) and build `node_id_map` to
translate Trackastra's internal graph node keys to submission IDs when writing edges.

### stderr suppression

`contextlib.redirect_stderr` captures all output to stderr during `model.track()`.
This silences Trackastra's internal tqdm progress bars without requiring any
knowledge of which internal modules use tqdm or how they call it.

### Tracking config + post-processing

Three knobs, all chosen so the defaults cannot degrade the score versus the raw
graph output:

- **`PRUNE_ISOLATED`** (default `True`): drop nodes with no incident edge. Trackastra
  emits one node per DoG detection; a node the tracker could not link to any other
  frame is almost always a false-positive detection. The metric is node-precision
  heavy **and** penalises over-prediction, so these orphans cost score twice. Pruning
  removes only zero-degree nodes — it never touches a real track or a division node
  (which by definition has ≥1 edge). This mirrors the baseline notebook (which scores
  higher partly because of it).
- **`TOPK_PER_FRAME`** (default `None` = off): cap detections per frame to the top-k by
  DoG response, curbing over-detection *before* tracking. Left off by default so it
  can only ever be an opt-in tightening, never a silent regression.
- **`TRACK_MODE`** (default `"greedy"`): `"ilp"` solves the assignment globally
  (better links, much slower — watch the 12 h Kaggle limit).

In [11]:
MASK_MODE = "watershed"  # "watershed" (real masks, appearance features) or "ball" (fixed spheres)
PRUNE_ISOLATED = True    # drop zero-degree nodes — almost all are false positives
TOPK_PER_FRAME = None    # cap detections/frame by DoG score (None = keep all)
TRACK_MODE = "greedy"    # "greedy" (fast) or "ilp" (global, better, slower)


@dataclass(frozen=True, slots=True)
class NodeRow:
    """One detected cell centroid at one timepoint."""

    dataset: str
    node_id: int
    t: int
    z: int
    y: int
    x: int


@dataclass(frozen=True, slots=True)
class EdgeRow:
    """Temporal link between two detections."""

    dataset: str
    source_id: int
    target_id: int


def track_one_dataset(
    zarr_path: Path,
    id_offset: int = 0,
    topk: int | None = TOPK_PER_FRAME,
    mode: str = TRACK_MODE,
) -> tuple[list[NodeRow], list[EdgeRow], int]:
    """Detect → mask → Trackastra track → prune, for one zarr volume.

    Node IDs are assigned as ``id_offset + 1, id_offset + 2, ...`` over *all* graph
    nodes (before pruning), so IDs never collide across datasets even after pruning
    removes some of them.

    Args:
        zarr_path: Path to the .zarr volume.
        id_offset: Add to every local node ID for global uniqueness.
        topk: Optional per-frame detection cap by DoG score.
        mode: Trackastra tracking mode ("greedy" or "ilp").

    Returns:
        (nodes, edges, n_assigned) — n_assigned is the count of IDs consumed
        (pre-prune), which the caller adds to its running offset.
    """
    dataset = zarr_path.stem
    n_t, n_z, n_y, n_x = get_volume_shape(zarr_path)
    n_y_iso = n_y // XY_DOWNSAMPLE
    n_x_iso = n_x // XY_DOWNSAMPLE
    shape_zyx = (n_z, n_y, n_x)
    shape_iso = (n_z, n_y_iso, n_x_iso)

    det_lookup: dict[tuple[int, int], tuple[float, float, float]] = {}
    imgs_4d = np.zeros((n_t, n_z, n_y_iso, n_x_iso), dtype=np.float32)
    masks_4d = np.zeros((n_t, n_z, n_y_iso, n_x_iso), dtype=np.uint16)

    for t in tqdm(range(n_t), desc=f"  {dataset} frames", leave=False):
        vol = load_timepoint(zarr_path, t, shape_zyx)
        coords, _scores = detect_peaks_dog(vol, topk=topk)
        imgs_4d[t] = make_isotropic(vol)
        centroids_orig = [(float(c[0]), float(c[1]), float(c[2])) for c in coords]
        if MASK_MODE == "watershed":
            masks_4d[t] = centroids_to_seg_mask(imgs_4d[t], centroids_orig)
        else:
            masks_4d[t] = centroids_to_mask(shape_iso, centroids_orig)
        for cell_id, (z_o, y_o, x_o) in enumerate(centroids_orig, start=1):
            det_lookup[(t, cell_id)] = (z_o, y_o, x_o)

    with contextlib.redirect_stderr(io.StringIO()):
        track_graph, _masks_tracked = model.track(
            imgs_4d.astype(np.uint16), masks_4d, mode=mode
        )

    node_id_map: dict = {}
    nodes: list[NodeRow] = []
    nid = id_offset + 1
    for graph_node, attrs in track_graph.nodes(data=True):
        centroid = det_lookup.get((int(attrs["time"]), int(attrs["label"])))
        if centroid is None:
            continue  # ghost node not in our mask (shouldn't happen)
        z_o, y_o, x_o = centroid
        node_id_map[graph_node] = nid
        nodes.append(
            NodeRow(dataset, nid, int(attrs["time"]), int(round(z_o)), int(round(y_o)), int(round(x_o)))
        )
        nid += 1
    n_assigned = len(node_id_map)

    edges: list[EdgeRow] = []
    for src, dst in track_graph.edges():
        s, d = node_id_map.get(src), node_id_map.get(dst)
        if s is not None and d is not None:
            edges.append(EdgeRow(dataset, s, d))

    if PRUNE_ISOLATED and edges:
        used = {e.source_id for e in edges} | {e.target_id for e in edges}
        nodes = [n for n in nodes if n.node_id in used]

    return nodes, edges, n_assigned

### Local proxy validation (read-only)

Score the full detect → track → prune pipeline on a couple of train movies against
their ground-truth `.geff` graphs, using a competition-style proxy metric
(`0.5·node_F1 + 0.4·edge_Jaccard + 0.1`). This never writes the submission — it exists
purely so parameter changes can be compared **without** spending a Kaggle submission.
The train folder ships with GT `.geff` graphs alongside each `.zarr`, so we score a
couple of embryo-diverse movies directly.

In [12]:
PROXY_GATE_UM = 7.0      # metric matching window (µm)
PROXY_VAL_SAMPLES = 2    # embryo-diverse train movies to score


def _read_geff_gt(geff_path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load GT nodes + edges from a .geff zarr store as DataFrames."""
    import zarr as _zarr

    g = _zarr.open(str(geff_path), mode="r")
    gt_nodes = pd.DataFrame({
        "node_id": np.asarray(g["nodes/ids"]),
        "t": np.asarray(g["nodes/props/t/values"]),
        "z": np.asarray(g["nodes/props/z/values"]),
        "y": np.asarray(g["nodes/props/y/values"]),
        "x": np.asarray(g["nodes/props/x/values"]),
    })
    e = np.asarray(g["edges/ids"])
    gt_edges = (
        pd.DataFrame({"source_id": e[:, 0], "target_id": e[:, 1]})
        if e.ndim == 2 and len(e)
        else pd.DataFrame({"source_id": pd.Series(dtype=int), "target_id": pd.Series(dtype=int)})
    )
    return gt_nodes, gt_edges


def proxy_score(nodes: list[NodeRow], edges: list[EdgeRow], geff_path: Path) -> tuple[float | None, dict]:
    """Proxy metric vs GT: 0.5·node_F1 + 0.4·edge_Jaccard + 0.1 (per-frame Hungarian match)."""
    gt_nodes, gt_edges = _read_geff_gt(geff_path)
    pred = pd.DataFrame([{"node_id": n.node_id, "t": n.t, "z": n.z, "y": n.y, "x": n.x} for n in nodes])
    if not len(pred):
        return None, {}
    gt_nodes = gt_nodes[gt_nodes["t"] <= int(pred["t"].max())]

    p2g: dict[int, int] = {}
    for t in sorted(set(pred["t"]) & set(gt_nodes["t"])):
        p = pred[pred["t"] == t].reset_index(drop=True)
        gg = gt_nodes[gt_nodes["t"] == t].reset_index(drop=True)
        if not len(p) or not len(gg):
            continue
        p_um = p[["z", "y", "x"]].values * SCALE_ZYX[None, :]
        g_um = gg[["z", "y", "x"]].values * SCALE_ZYX[None, :]
        dist = np.sqrt(((p_um[:, None] - g_um[None]) ** 2).sum(2))
        cost = np.where(dist <= PROXY_GATE_UM, dist, 1e6)
        ri, ci = linear_sum_assignment(cost)
        for a, b in zip(ri, ci, strict=False):
            if cost[a, b] < 1e6:
                p2g[int(p.loc[a, "node_id"])] = int(gg.loc[b, "node_id"])

    tp = len(p2g)
    prec, rec = tp / max(len(pred), 1), tp / max(len(gt_nodes), 1)
    node_f1 = 2 * prec * rec / max(prec + rec, 1e-9)

    valid = set(gt_nodes["node_id"])
    gt_edges = gt_edges[gt_edges["source_id"].isin(valid) & gt_edges["target_id"].isin(valid)]
    gt_eset = set(zip(gt_edges["source_id"].astype(int), gt_edges["target_id"].astype(int), strict=False))
    pred_mapped = {(p2g[e.source_id], p2g[e.target_id]) for e in edges if e.source_id in p2g and e.target_id in p2g}
    etp = len(pred_mapped & gt_eset)
    ep, er = etp / max(len(pred_mapped), 1), etp / max(len(gt_eset), 1)
    edge_f1 = 2 * ep * er / max(ep + er, 1e-9)

    score = round(0.5 * node_f1 + 0.4 * edge_f1 + 0.1, 4)
    return score, {"node_f1": round(node_f1, 3), "edge_f1": round(edge_f1, 3), "pred": len(pred), "gt": len(gt_nodes)}


assert TRAIN_DIR.exists(), f"Train dir missing: {TRAIN_DIR}"

# one movie per embryo prefix, capped at PROXY_VAL_SAMPLES
_picks: dict[str, Path] = {}
for _zp in sorted(TRAIN_DIR.glob("*.zarr")):
    _picks.setdefault(_zp.stem.split("_")[0], _zp)
_picks = dict(list(_picks.items())[:PROXY_VAL_SAMPLES])

for _zp in _picks.values():
    _geff = TRAIN_DIR / (_zp.stem + ".geff")
    _n, _e, _ = track_one_dataset(_zp)
    _sc, _br = proxy_score(_n, _e, _geff)
    print(f"  {_zp.stem[:28]:28s} proxy={_sc}  {_br}")

  44b6_0113de3b frames:   0%|          | 0/100 [00:00<?, ?it/s]

  44b6_0113de3b                proxy=0.4477  {'node_f1': 0.005, 'edge_f1': 0.864, 'pred': 22570, 'gt': 52}


  6bba_05b6850b frames:   0%|          | 0/100 [00:00<?, ?it/s]

  6bba_05b6850b                proxy=0.566  {'node_f1': 0.213, 'edge_f1': 0.898, 'pred': 7024, 'gt': 861}


### Run all test datasets

In [13]:
all_node_rows: list[NodeRow] = []
all_edge_rows: list[EdgeRow] = []

test_zarr_paths = sorted(TEST_DIR.glob("*.zarr"))
assert test_zarr_paths, f"No .zarr files found in {TEST_DIR}"
print(f"Test datasets: {len(test_zarr_paths)}")

_offset = 0
for zarr_path in tqdm(test_zarr_paths, desc="datasets"):
    nodes, edges, n_assigned = track_one_dataset(zarr_path, id_offset=_offset)
    _offset += n_assigned  # advance by IDs consumed (pre-prune) — avoids cross-dataset collisions
    all_node_rows += nodes
    all_edge_rows += edges
    print(f"  {zarr_path.stem}: {len(nodes)} nodes, {len(edges)} edges, "
          f"ratio={len(edges)/max(len(nodes),1):.2f}")

Test datasets: 4


datasets:   0%|          | 0/4 [00:00<?, ?it/s]

  44b6_0113de3b frames:   0%|          | 0/100 [00:00<?, ?it/s]

  44b6_0113de3b: 22570 nodes, 19855 edges, ratio=0.88


  44b6_0b24845f frames:   0%|          | 0/100 [00:00<?, ?it/s]

  44b6_0b24845f: 10965 nodes, 7423 edges, ratio=0.68


  6bba_05b6850b frames:   0%|          | 0/100 [00:00<?, ?it/s]

  6bba_05b6850b: 7024 nodes, 6770 edges, ratio=0.96


  6bba_05db0fb1 frames:   0%|          | 0/100 [00:00<?, ?it/s]

  6bba_05db0fb1: 29105 nodes, 20503 edges, ratio=0.70


## Build Submission CSV

Write all node rows first, then all edge rows. The `id` column is a consecutive
integer index with no semantic meaning — the scorer ignores it.

**Sanity checks before submitting:**
- Every test dataset name appears in the `dataset` column
- Edge `source_id` and `target_id` all reference valid `node_id` values
- `z`, `y`, `x` are within the volume bounds (no negative coords, no out-of-bounds)
- Edge/node ratio: expect ~(T−1)/T per dataset (≥95% for T≥20). Much lower = tracking gaps.

In [ ]:
import sys

# biohub_cellops must be attached as an offline Kaggle dataset.
_cellops_modules = list(Path("/kaggle/input").rglob("biohub_cellops/submission_guard.py"))
assert _cellops_modules, "Attached inputs do not contain the biohub_cellops package"
_cellops_root = _cellops_modules[0].parents[1]
if str(_cellops_root) not in sys.path:
    sys.path.insert(0, str(_cellops_root))

from biohub_cellops.submission_guard import KaggleSubmissionCompiler
from biohub_cellops.trackastra_adapter import TrackastraCellOpsAdapter

cells, links = TrackastraCellOpsAdapter.adapt(all_node_rows, all_edge_rows)
submission_rows = KaggleSubmissionCompiler.compile(cells, links)
dataset_shapes = {path.stem: get_volume_shape(path) for path in test_zarr_paths}
validation_report = KaggleSubmissionCompiler.validate_dataset_context(
    submission_rows, dataset_shapes
)

sample_path = next(Path("/kaggle/input").rglob("sample_submission.csv"))
sub_path = Path("/kaggle/working/submission.csv")
KaggleSubmissionCompiler.write_csv(
    submission_rows,
    sub_path,
    sample_submission_path=sample_path,
    dataset_shapes=dataset_shapes,
)

print(f"Written and re-read validated: {sub_path}")
print(f"Total rows: {len(submission_rows)}")
for dataset, report in validation_report.items():
    print(dataset, report)

## Sanity Check

### What to look for

**Row counts:**
- `node` count = total cell detections across all datasets and timepoints
- `edge` count should be close to `node` count (ideally within 5%)
- Large gap = cells detected but not tracked → hurts Edge Jaccard

**Edge/node ratio per dataset** (printed in inference loop):
- Expected ≈ (T−1)/T ≈ 0.98 for T=50 frames
- Ratio < 0.90 = many tracking failures; check DoG threshold or try `mode="ilp"`

**Orphan analysis** — nodes with no outgoing edge at non-last timepoints:
```python
edge_sources = set(df_sub[df_sub.row_type=='edge'].source_id)
nodes = df_sub[df_sub.row_type=='node']
orphans = nodes[~nodes.node_id.isin(edge_sources)]
print(orphans['t'].value_counts().sort_index())
# Expect spike at last timepoint only; flat distribution elsewhere = over-detection
```

**Dataset coverage:** all test dataset names must appear — missing = score 0.

In [ ]:
node_rows = [row for row in submission_rows if row["row_type"] == "node"]
edge_rows = [row for row in submission_rows if row["row_type"] == "edge"]
print(f"nodes={len(node_rows)} edges={len(edge_rows)} datasets={len(validation_report)}")
print(*submission_rows[:4], sep="\n")

## Next Step: Fine-tune Trackastra on Zebrafish

The pretrained `ctc` checkpoint never saw zebrafish embryos — it generalises from other
Cell Tracking Challenge datasets. Combined with our fabricated masks, that domain gap is
the main reason this pipeline trails a well-tuned classical linker. Real watershed masks
(above) close part of it; **fine-tuning closes the rest** and has the highest ceiling.

**Plan (high effort — multi-day, GPU required):**

1. **Convert train `.geff` → Trackastra training format.** Trackastra trains on
   `(image, instance-mask, tracklet-graph)` triples in CTC-style layout. For each train
   movie: build per-frame instance masks (the same watershed seeding used here, but seeded
   from the **GT** centroids in the `.geff`), and emit the GT lineage as the target graph.
2. **Fine-tune from the `ctc` checkpoint** rather than training from scratch — far fewer
   epochs, and the learned association features transfer. Hold out ≥1 embryo for validation.
3. **Validate with the proxy scorer above** (`proxy_score`) on the held-out embryo before
   every checkpoint promotion — never promote on train loss alone.
4. **Bundle the fine-tuned weights** exactly like `models/ctc/` (Setup Step 3) and point
   `MODEL_DIR` at them for the offline submission.

> ⚠ **Verify the training API first.** The exact entrypoint (module/CLI, config schema,
> expected on-disk layout) must be read from the installed `trackastra` version before
> writing any conversion code — do not assume it from memory. Start here:
> ```
> python -c "import trackastra, inspect; print(trackastra.__version__)"
> python -c "import trackastra.training as tt; help(tt)"   # confirm module + signatures
> ```
> Cross-check against the repo docs: <https://github.com/weigertlab/trackastra#training>.

**Expected payoff vs cost:** highest score ceiling of all options, but days of work and a
GPU budget. Only worth starting once watershed masks + ILP mode have been measured and the
classical baseline has genuinely plateaued.

In [ ]:
! head -5 {sub_path}